# Emotions of Place: A Comparative Visual Essay
## [Essay Title — Describe the story your maps tell]
### [Names] &nbsp;|&nbsp; [Institution compared] &nbsp;|&nbsp; [Date]

<style>
/* Hide code cells for clean essay presentation — remove this cell to show code */
.jp-CodeCell .jp-InputArea { display: none !important; }
div.code_cell div.input_area { display: none !important; }
.jp-Notebook { width: 80% !important; max-width: 1100px !important; margin: 0 auto !important; }
</style>

<div style="background-color: #e3f2fd; border: 2px solid #1976d2; border-radius: 10px; padding: 20px; margin: 10px 0;">

### 📋 Assignment Overview

Over the last several weeks you reproduced the workflow from Ryan Heuser's ["The Emotions of London"](https://litlab.stanford.edu/LiteraryLabPamphlet13.pdf): you scraped Reddit data for JMU and a second Virginia institution, geoparsed the posts for place names, corrected the geoparser's mistakes, and attached RoBERTa sentiment scores to each sentence.

This visual essay is the first deliverable for the mapping project. Its purpose is to tell a **comparative story**: what do the two datasets reveal about how students at each institution talk about the places around them? The essay moves through four stages:

1. **Introduction** — state your hypothesis and describe the two corpora
2. **Data Cleaning Analysis** — document what the geoparser got wrong and what you fixed
3. **Comparative Sentiment Maps** — apply consistent map design decisions to both datasets
4. **Close Reading** — examine three standout data points from each dataset in depth

**When you are finished:** delete all instruction cells (the ones with colored backgrounds or labeled ✍️) and leave only your writing, the code outputs, and the maps.

</div>

In [ ]:
# ── Libraries ────────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.colors as pc
import mapclassify
import numpy as np

In [ ]:
# ── Load JMU sentiment data ──────────────────────────────────────────────────
# This is the sentence-level data your team produced in Lesson 5.2.
# If the team file is missing, the backup is loaded automatically.

try:
    df_jmu_raw = pd.read_pickle('data/jmu_reddit_sentiment_full.pickle')
    print(f"✅ JMU team file loaded: {len(df_jmu_raw):,} rows")
except FileNotFoundError:
    df_jmu_raw = pd.read_pickle('data/JMU/JMU_geoparsed_long_backup_sentiment.pickle')
    print(f"⚠️  JMU backup loaded: {len(df_jmu_raw):,} rows")

# Apply review-sheet corrections (Lesson 4 workflow)
_df = df_jmu_raw.copy()
_review_cols = {'action', 'corrected_name', 'corrected_latlon', 'corrected_place_type'}
if _review_cols.issubset(_df.columns):
    _df = _df[_df['action'] != 'REMOVE'].reset_index(drop=True)
    _mask = _df['action'] == 'CORRECT'
    _has_name = _mask & _df['corrected_name'].fillna('').str.strip().ne('')
    _df.loc[_has_name, 'place'] = _df.loc[_has_name, 'corrected_name']
    _has_coords = _mask & _df['corrected_latlon'].fillna('').str.strip().ne('')
    if _has_coords.any():
        _coords = _df.loc[_has_coords, 'corrected_latlon'].str.split(',', expand=True)
        _df.loc[_has_coords, 'latitude']  = pd.to_numeric(_coords[0].str.strip(), errors='coerce')
        _df.loc[_has_coords, 'longitude'] = pd.to_numeric(_coords[1].str.strip(), errors='coerce')
    _has_type = _mask & _df['corrected_place_type'].fillna('').str.strip().ne('')
    _df.loc[_has_type, 'place_type'] = _df.loc[_has_type, 'corrected_place_type']

# Aggregate to place-level
df_jmu_places = (
    _df
    .dropna(subset=['place', 'latitude', 'longitude'])
    .astype({'latitude': float, 'longitude': float})
    .groupby('place', sort=False)
    .agg(
        location_count=('place', 'size'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first'),
        sentences=('sentences', lambda x: ' | '.join(str(s) for s in list(x)[:5])),
        avg_roberta_compound=('roberta_compound', 'mean'),
    )
    .reset_index()
)
if 'place_type' in _df.columns:
    _ref = (
        _df.loc[_df['place_type'].fillna('').ne(''), ['place', 'place_type']]
        .drop_duplicates('place')
    )
    df_jmu_places = df_jmu_places.merge(_ref, on='place', how='left')

print(f"📍 JMU: {len(df_jmu_places):,} unique places after aggregation")

In [ ]:
# ── Load institution data ────────────────────────────────────────────────────
# ✏️ TO DO: update the group number and institution name below
INSTITUTION_NAME = "[Your Institution]"  # e.g. 'GMU', 'VCU', 'ODU'

df_institution_raw = pd.read_csv(
    "group_data_packets/group_N/python/INSTITUTION_processed_clean.csv"  # ← update this path
)
print(f"✅ {INSTITUTION_NAME} data loaded: {len(df_institution_raw):,} rows")

# Standardize: apply revised values, remove false positives
_inst = df_institution_raw.copy()
if 'false_positive' in _inst.columns:
    _inst = _inst[_inst['false_positive'].ne(True)].copy()
if 'revised_place' in _inst.columns:
    _mask = _inst['revised_place'].fillna('').str.strip().ne('')
    _inst.loc[_mask, 'place'] = _inst.loc[_mask, 'revised_place']
if 'revised_latitude' in _inst.columns:
    _mask = _inst['revised_latitude'].notna()
    _inst.loc[_mask, 'latitude'] = pd.to_numeric(_inst.loc[_mask, 'revised_latitude'], errors='coerce')
if 'revised_longitude' in _inst.columns:
    _mask = _inst['revised_longitude'].notna()
    _inst.loc[_mask, 'longitude'] = pd.to_numeric(_inst.loc[_mask, 'revised_longitude'], errors='coerce')

# Aggregate to place-level
df_institution_places = (
    _inst
    .dropna(subset=['place', 'latitude', 'longitude'])
    .astype({'latitude': float, 'longitude': float})
    .groupby('place', sort=False)
    .agg(
        location_count=('place', 'size'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first'),
        sentences=('sentences', lambda x: ' | '.join(str(s) for s in list(x)[:5])),
        avg_roberta_compound=('roberta_compound', 'mean'),
    )
    .reset_index()
)
if 'place_type' in _inst.columns:
    _ref = (
        _inst.loc[_inst['place_type'].fillna('').ne(''), ['place', 'place_type']]
        .drop_duplicates('place')
    )
    df_institution_places = df_institution_places.merge(_ref, on='place', how='left')

print(f"📍 {INSTITUTION_NAME}: {len(df_institution_places):,} unique places after aggregation")

---
## 1 Introduction

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — Corpus Description**

Skim both datasets before writing. Compare the two institutions across at least three of the following dimensions: size of corpus, date range, geographic spread of mentions, or dominant place types. Keep this to one paragraph.

*Delete this instruction cell before submitting.*
</div>

### 1.1 Corpus Description

*Write your corpus description here.*

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — Hypothesis**

State a working hypothesis about how the spatial sentiment of the two institutions differs. Your hypothesis must be grounded in **both space and sentiment** — for example: "JMU Reddit posts will show more positive sentiment about on-campus buildings than GMU posts, which will skew toward commuter-related infrastructure." One paragraph.

*Delete this instruction cell before submitting.*
</div>

### 1.2 Hypothesis

*State your hypothesis here.*

---
## 2 Data Cleaning Analysis

Every location in these datasets passed through an NER model and a geoparser before reaching this notebook. Neither tool is infallible. In Lesson 4 you audited the geoparsed output and corrected the most significant errors. This section documents what you found and what you changed — it is the **methods** section of your paper.

### 2.1 JMU — Review Sheet Corrections

In [ ]:
# ── JMU review sheet: show what was corrected and removed ───────────────────
_review_cols = {'action', 'corrected_name', 'corrected_latlon', 'corrected_place_type'}
if _review_cols.issubset(df_jmu_raw.columns):
    _corrected = (
        df_jmu_raw[df_jmu_raw['action'] == 'CORRECT']
        [['place', 'corrected_name', 'corrected_latlon', 'corrected_place_type']]
        .drop_duplicates('place')
        .reset_index(drop=True)
    )
    _removed = (
        df_jmu_raw[df_jmu_raw['action'] == 'REMOVE']
        [['place']]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    print(f"CORRECT: {len(_corrected)} places renamed or relocated")
    print(f"REMOVE:  {len(_removed)} false positives removed")
    print("\n── Corrections ──")
    display(_corrected)
    print("\n── Removed (false positives) ──")
    display(_removed)
else:
    print("No review columns found in JMU dataset — corrections were not tracked in this version.")

In [ ]:
# ── JMU post-correction location map ────────────────────────────────────────
fig = px.scatter_map(
    df_jmu_places,
    lat='latitude', lon='longitude',
    size='location_count',
    color='place_type' if 'place_type' in df_jmu_places.columns else 'avg_roberta_compound',
    hover_name='place',
    hover_data={'location_count': True, 'avg_roberta_compound': ':.3f',
                'latitude': False, 'longitude': False},
    size_max=20,
    map_style='carto-positron',
    center={'lat': 38.4, 'lon': -78.9}, zoom=7, height=480,
    title='JMU Reddit — Corrected Locations'
)
fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig.show()

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — JMU Corrections**

Describe the major errors you found in the JMU geoparsed data and explain why they occurred. Consider: Which place names were resolved to the wrong coordinates? Which NER extractions were not actually places? What do these errors reveal about how geoparsers make decisions? One to two paragraphs.

> 👉 **Note:** *The geoparser resolves ambiguous names by population rank, so a common English word that is also a place name (e.g. "Valley") will almost always be resolved to the largest city with that name, not the local feature the author intended.*

*Delete this instruction cell before submitting.*
</div>

*Write your analysis of the JMU corrections here.*

### 2.2 [Institution] — Review Sheet Corrections

In [ ]:
# ── Institution review sheet: show what was corrected and removed ────────────
if 'revised_place' in df_institution_raw.columns:
    _revised = (
        df_institution_raw[
            df_institution_raw['revised_place'].fillna('').str.strip().ne('')
        ]
        [['place', 'revised_place', 'revised_latitude', 'revised_longitude', 'place_type']]
        .drop_duplicates('place')
        .reset_index(drop=True)
    )
    _fp = (
        df_institution_raw[df_institution_raw['false_positive'].eq(True)]
        [['place', 'sentences']]
        .drop_duplicates('place')
        .reset_index(drop=True)
    )
    print(f"Revised:         {len(_revised)} place records corrected")
    print(f"False positives: {len(_fp)} places removed")
    print("\n── Corrections ──")
    display(_revised)
    print("\n── Removed (false positives) ──")
    display(_fp)
else:
    print("No revision columns found — check the CSV file structure.")

In [ ]:
# ── Institution post-correction location map ─────────────────────────────────
# ✏️ TO DO: adjust center coordinates and zoom for your institution's region
fig = px.scatter_map(
    df_institution_places,
    lat='latitude', lon='longitude',
    size='location_count',
    color='place_type' if 'place_type' in df_institution_places.columns else 'avg_roberta_compound',
    hover_name='place',
    hover_data={'location_count': True, 'avg_roberta_compound': ':.3f',
                'latitude': False, 'longitude': False},
    size_max=20,
    map_style='carto-positron',
    center={'lat': 37.5, 'lon': -78.0}, zoom=6, height=480,  # ← update center
    title=f'{INSTITUTION_NAME} Reddit — Corrected Locations'
)
fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig.show()

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — [Institution] Corrections**

Describe the major errors in the institution dataset and what you corrected. Were the errors similar to JMU's? What does the pattern of errors suggest about the linguistic context of that institution's Reddit community? One to two paragraphs.

*Delete this instruction cell before submitting.*
</div>

*Write your analysis of the institution corrections here.*

---
## 3 Comparative Sentiment Maps

The same map design decisions you explored in Lesson 6 must be applied consistently to both datasets. Using different parameters for each map would introduce visual distortion that could make the comparison misleading. Set your decisions once in the cell below and both maps will use them.

### 3.1 Design Brief

Complete this table **before running the code cells below**. Every parameter must be a deliberate choice with a stated rationale.

| Design Decision | Your Choice | Reasoning |
|---|---|---|
| **Filtering threshold** (min post count) | ___ | |
| **Place types** | ___ | |
| **Size classes** (Jenks) | ___ | |
| **Color buckets** (Jenks) | ___ | |
| **Color scale** | ___ | |
| **Base map style** | ___ | |
| **Zoom / center** | Zoom: ___ · Lat: ___ · Lon: ___ | |

> 👉 **Note:** *Both maps must use identical parameter values. Any difference in settings is a claim, not a style choice.*

In [ ]:
# ── Shared design decisions — apply to BOTH maps ─────────────────────────────
MIN_COUNT       = 3                                              # Decision 1
PLACE_TYPES     = ['City', 'Building', 'University', 'Neighborhood']  # Decision 2
N_SIZE_CLASSES  = 4                                              # Decision 3b
N_COLOR_BUCKETS = 5                                              # Decision 4
COLOR_SCALE     = 'RdYlGn'                                      # Decision 5
MAP_STYLE       = 'carto-positron'                              # Decision 6
CENTER          = {'lat': 37.5, 'lon': -78.0}                   # Decision 7
ZOOM            = 6                                             # Decision 7


def _build_sentiment_map(df_places, title):
    """Filter, classify, and map sentiment for one dataset."""
    if PLACE_TYPES is not None:
        df_work = df_places[
            (df_places['location_count'] >= MIN_COUNT) &
            df_places['place_type'].isin(PLACE_TYPES)
        ].copy()
    else:
        df_work = df_places[
            df_places['location_count'] >= MIN_COUNT
        ].copy()

    # Jenks size classification
    _jnb_s = mapclassify.NaturalBreaks(df_work['location_count'].values, k=N_SIZE_CLASSES)
    df_work['size_class'] = (_jnb_s.yb + 1).astype(float)

    # Jenks color classification
    scores  = df_work['avg_roberta_compound']
    _jnb_c  = mapclassify.NaturalBreaks(scores.values, k=N_COLOR_BUCKETS)
    _breaks = _jnb_c.bins
    _lo     = scores.min()
    _labels = []
    for _hi in _breaks:
        _labels.append(f"{_lo:.2f} to {_hi:.2f}")
        _lo = _hi
    df_work['color_class'] = pd.cut(scores, bins=[-float('inf')] + list(_breaks), labels=_labels)
    _palette   = pc.sample_colorscale(COLOR_SCALE, [i / (N_COLOR_BUCKETS - 1) for i in range(N_COLOR_BUCKETS)])
    _color_map = dict(zip(_labels, _palette))

    fig = px.scatter_map(
        df_work,
        lat='latitude', lon='longitude',
        size='size_class', color='color_class',
        hover_name='place',
        hover_data={'avg_roberta_compound': ':.3f', 'location_count': True,
                    'size_class': False, 'color_class': False,
                    'latitude': False, 'longitude': False},
        color_discrete_map=_color_map,
        category_orders={'color_class': _labels},
        size_max=18, map_style=MAP_STYLE,
        center=CENTER, zoom=ZOOM, height=520,
        title=title
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
    return fig

print(f"Design decisions set: MIN_COUNT={MIN_COUNT}, PLACE_TYPES={PLACE_TYPES}, "
      f"N_SIZE_CLASSES={N_SIZE_CLASSES}, N_COLOR_BUCKETS={N_COLOR_BUCKETS}, "
      f"COLOR_SCALE='{COLOR_SCALE}', MAP_STYLE='{MAP_STYLE}'")

### 3.2 JMU Sentiment Map

In [ ]:
_build_sentiment_map(df_jmu_places, 'JMU Reddit — Average Sentiment by Place').show()

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — JMU Map**

Describe the overall spatial pattern. Where are the most frequently mentioned places? Are they positive, negative, or neutral? Does any place surprise you? One paragraph.

*Delete this instruction cell before submitting.*
</div>

*Write your analysis of the JMU map here.*

### 3.3 [Institution] Sentiment Map

In [ ]:
_build_sentiment_map(df_institution_places, f'{INSTITUTION_NAME} Reddit — Average Sentiment by Place').show()

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — [Institution] Map**

Describe the spatial pattern for the institution dataset using the same framework as the JMU map. One paragraph.

*Delete this instruction cell before submitting.*
</div>

*Write your analysis of the institution map here.*

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — Map Comparison**

Compare the two maps directly. What is the most significant spatial or emotional difference between them? Does this confirm, complicate, or contradict your hypothesis? What limitations in the data or the design decisions might be distorting the picture? One to two paragraphs.

*Delete this instruction cell before submitting.*
</div>

### 3.4 Comparative Analysis

*Write your comparative map analysis here.*

---
## 4 Close Reading — Three Standout Data Points

A map compresses thousands of sentences into a single colored dot. This section zooms in. For each dataset you will select three places that stood out — either because of an unexpected sentiment score, a surprisingly high mention count, or a tension between what the place represents and how it was discussed. Examining the actual sentences restores the human texture that aggregation erases.

### 4.1 JMU — Three Standout Places

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — Introducing the Three Places**

Before analyzing each place, write one or two sentences introducing the three you have chosen and explaining why they stood out when you examined the map.

*Delete this instruction cell before submitting.*
</div>

*Introduce your three standout JMU places here.*

In [ ]:
# ── Set the three JMU places to analyze ─────────────────────────────────────
# ✏️ TO DO: replace with the actual place names from your dataset
JMU_PLACES = [
    "Harrisonburg",   # ← standout place 1
    "D-Hall",         # ← standout place 2
    "Valley Mall",    # ← standout place 3
]

# Display sentence-level data for each place
for place_name in JMU_PLACES:
    _rows = df_jmu_raw[
        df_jmu_raw['place'].str.lower() == place_name.lower()
    ][['sentences', 'roberta_compound']].dropna().head(10)

    _agg = df_jmu_places[df_jmu_places['place'].str.lower() == place_name.lower()]
    if len(_agg):
        _count   = _agg['location_count'].values[0]
        _sent    = _agg['avg_roberta_compound'].values[0]
        print(f"\n── {place_name}  |  {_count} mentions  |  avg sentiment = {_sent:.3f} ──")
    else:
        print(f"\n── {place_name}  |  (not found in aggregated data — check spelling) ──")

    display(
        _rows.style
        .background_gradient(subset=['roberta_compound'], cmap='RdYlGn', vmin=-1, vmax=1)
        .format({'roberta_compound': '{:.3f}'})
        .set_caption(place_name)
    )

In [ ]:
# ── Focused map: highlight the three standout JMU places ────────────────────
df_jmu_focus = df_jmu_places[
    df_jmu_places['location_count'] >= MIN_COUNT
].copy()
df_jmu_focus['highlight'] = df_jmu_focus['place'].isin(JMU_PLACES)

fig = px.scatter_map(
    df_jmu_focus,
    lat='latitude', lon='longitude',
    size='location_count',
    color='avg_roberta_compound',
    hover_name='place',
    hover_data={'location_count': True, 'avg_roberta_compound': ':.3f',
                'latitude': False, 'longitude': False},
    color_continuous_scale=COLOR_SCALE, color_continuous_midpoint=0,
    size_max=20, map_style=MAP_STYLE,
    center=CENTER, zoom=ZOOM, height=480,
    title='JMU — Standout Places in Context'
)
# Add annotation markers for the selected places
for _, row in df_jmu_focus[df_jmu_focus['highlight']].iterrows():
    fig.add_scattermap(
        lat=[row['latitude']], lon=[row['longitude']],
        mode='markers+text',
        marker=dict(size=14, color='black', opacity=0.6),
        text=[row['place']], textposition='top right',
        textfont=dict(size=11, color='black'),
        showlegend=False
    )
fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig.show()

#### Place 1 — [Name]

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task**

Analyze the sentences for this place. What is the dominant tone? Are there any sentences where the RoBERTa score seems wrong — either too positive or too negative? What does the sentiment reveal about how JMU students relate to this place? One paragraph per place.

*Delete this instruction cell before submitting.*
</div>

*Write your analysis of JMU Place 1 here.*

#### Place 2 — [Name]

*Write your analysis of JMU Place 2 here.*

#### Place 3 — [Name]

*Write your analysis of JMU Place 3 here.*

### 4.2 [Institution] — Three Standout Places

*Introduce your three standout institution places here.*

In [ ]:
# ── Set the three institution places to analyze ──────────────────────────────
# ✏️ TO DO: replace with actual place names from your institution dataset
INST_PLACES = [
    "[Place 1]",   # ← standout place 1
    "[Place 2]",   # ← standout place 2
    "[Place 3]",   # ← standout place 3
]

for place_name in INST_PLACES:
    _rows = _inst[
        _inst['place'].str.lower() == place_name.lower()
    ][['sentences', 'roberta_compound']].dropna().head(10)

    _agg = df_institution_places[
        df_institution_places['place'].str.lower() == place_name.lower()
    ]
    if len(_agg):
        _count = _agg['location_count'].values[0]
        _sent  = _agg['avg_roberta_compound'].values[0]
        print(f"\n── {place_name}  |  {_count} mentions  |  avg sentiment = {_sent:.3f} ──")
    else:
        print(f"\n── {place_name}  |  (not found — check spelling) ──")

    display(
        _rows.style
        .background_gradient(subset=['roberta_compound'], cmap='RdYlGn', vmin=-1, vmax=1)
        .format({'roberta_compound': '{:.3f}'})
        .set_caption(place_name)
    )

In [ ]:
# ── Focused map: highlight the three standout institution places ─────────────
df_inst_focus = df_institution_places[
    df_institution_places['location_count'] >= MIN_COUNT
].copy()
df_inst_focus['highlight'] = df_inst_focus['place'].isin(INST_PLACES)

fig = px.scatter_map(
    df_inst_focus,
    lat='latitude', lon='longitude',
    size='location_count',
    color='avg_roberta_compound',
    hover_name='place',
    hover_data={'location_count': True, 'avg_roberta_compound': ':.3f',
                'latitude': False, 'longitude': False},
    color_continuous_scale=COLOR_SCALE, color_continuous_midpoint=0,
    size_max=20, map_style=MAP_STYLE,
    center=CENTER, zoom=ZOOM, height=480,
    title=f'{INSTITUTION_NAME} — Standout Places in Context'
)
for _, row in df_inst_focus[df_inst_focus['highlight']].iterrows():
    fig.add_scattermap(
        lat=[row['latitude']], lon=[row['longitude']],
        mode='markers+text',
        marker=dict(size=14, color='black', opacity=0.6),
        text=[row['place']], textposition='top right',
        textfont=dict(size=11, color='black'),
        showlegend=False
    )
fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig.show()

#### Place 1 — [Name]

*Write your analysis of Institution Place 1 here.*

#### Place 2 — [Name]

*Write your analysis of Institution Place 2 here.*

#### Place 3 — [Name]

*Write your analysis of Institution Place 3 here.*

### 4.3 Cross-Dataset Comparison

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — Cross-Dataset Comparison**

Compare the six places across the two datasets. Are there structural similarities in what kinds of places generate the strongest sentiment? Do the two communities discuss different *types* of places — or the same types but with different emotional registers? Does this comparison support your hypothesis? One to two paragraphs.

*Delete this instruction cell before submitting.*
</div>

*Write your cross-dataset comparison here.*

---
## 5 Conclusion

<div style="background-color: #fff8e1; border: 1px solid #f9a825; border-radius: 8px; padding: 16px; margin: 8px 0;">

✍️ **Writing Task — Conclusion**

Your conclusion should address all of the following:

- **Hypothesis verdict:** Did your maps and close reading confirm or contradict your hypothesis? Be specific.
- **Pipeline limitations:** Every step introduced noise — sentence splitting, NER errors, geoparser disambiguation, model bias in RoBERTa. Which limitation had the most impact on your analysis?
- **Design limitations:** What map design decisions did you make that could have told a different story? What would you do differently?
- **Future directions:** If you had more time or better data, how would you extend or improve this analysis?

Two to three paragraphs.

*Delete this instruction cell before submitting.*
</div>

*Write your conclusion here.*